In [ ]:
# ============================================================
# LUNG SOUND CLASSIFICATION - ADVANCED OPTIMIZATION (FIXED)
# Memory-optimized version for large datasets
# ============================================================
import os
import random
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, BatchNormalization, Dropout, LSTM
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
import gc

print("\n" + "="*120)
print("🫁 LUNG SOUND CLASSIFICATION - ADVANCED OPTIMIZATION (FIXED)")
print("="*120)

# ============================================================
# PARAMETRELER (OPTIMIZED)
# ============================================================
SEGMENT_SEC = 2.0
SR = 22050
N_MFCC = 40
NUM_CLASSES = 3
EPOCHS = 100
BATCH_SIZE = 16  # REDUCED from 32
RANDOM_STATE = 42
MAX_SAMPLES_PER_CLASS = 200  # LIMIT per class to avoid memory issues

# ============================================================
# 1. AUDIO PREPROCESSING
# ============================================================
print("\n📂 Adım 1: Ses Dosyalarını Yükle")
print("-"*120)

def advanced_noise_reduction(audio, sr):
    """Spektral çıkarma ile gürültü azaltma"""
    S = librosa.feature.melspectrogram(y=audio, sr=sr)
    S_db = librosa.power_to_db(S, ref=np.max)
    threshold = np.percentile(S_db, 35)
    mask = np.exp(-((threshold - S_db) ** 2) / (2 * (threshold / 3) ** 2))
    S_filtered = S * mask
    audio_filtered = librosa.feature.inverse.mel_to_audio(S_filtered, sr=sr)
    return np.clip(audio_filtered, -1, 1)

def advanced_normalization(audio):
    """Peak ve RMS normalization"""
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    rms = np.sqrt(np.mean(audio ** 2))
    if rms > 0:
        audio = audio / (rms + 1e-8)
    return audio

class_folders = {
    "Asthma": "./Datasets/Asthma",
    "COPD": "./Datasets/COPD4",
    "Healthy": "./Datasets/Healthy"
}

total_files = 0
for label, folder in class_folders.items():
    if os.path.exists(folder):
        files = len([f for f in os.listdir(folder) if f.endswith('.wav')])
        print(f"  📁 {label}: {files} dosya")
        total_files += files

if total_files == 0:
    print("\n❌ HATA: Hiç ses dosyası bulunamadı!")
    exit()

print(f"\n✅ Toplam {total_files} ses dosyası bulundu")

# ============================================================
# 2. FEATURE EXTRACTION
# ============================================================
def segment_audio(audio, sr, overlap_ratio=0.3):
    """Yüksek overlap ile segment"""
    win = int(SEGMENT_SEC * sr)
    hop = int(win * (1 - overlap_ratio))
    return [audio[i:i + win] for i in range(0, len(audio) - win + 1, hop)]

def extract_advanced_features(audio, sr=SR):
    """Kapsamlı feature extraction (280+ features)"""
    
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_fft=2048, hop_length=512)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)
    mfcc_min = mfcc.min(axis=1)
    mfcc_max = mfcc.max(axis=1)
    
    delta = librosa.feature.delta(mfcc)
    delta_mean = delta.mean(axis=1)
    delta_std = delta.std(axis=1)
    
    delta2 = librosa.feature.delta(mfcc, order=2)
    delta2_mean = delta2.mean(axis=1)
    delta2_std = delta2.std(axis=1)
    
    spec_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    spec_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)[0]
    spec_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)[0]
    zero_crossing = librosa.feature.zero_crossing_rate(audio)[0]
    
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    chroma_mean = chroma.mean(axis=1)
    
    rms = librosa.feature.rms(y=audio)[0]
    rms_mean = rms.mean()
    rms_std = rms.std()
    
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr)
    mel_mean = mel_spec.mean(axis=1)
    mel_std = mel_spec.std(axis=1)
    
    tempogram = librosa.feature.tempogram(y=audio, sr=sr)
    tempogram_mean = tempogram.mean(axis=1)
    
    features = np.hstack([
        mfcc_mean, mfcc_std, mfcc_min, mfcc_max,
        delta_mean, delta_std,
        delta2_mean, delta2_std,
        [spec_centroid.mean(), spec_centroid.std()],
        [spec_rolloff.mean(), spec_rolloff.std()],
        [spec_bandwidth.mean(), spec_bandwidth.std()],
        [zero_crossing.mean(), zero_crossing.std()],
        chroma_mean,
        tempogram_mean,
        [rms_mean, rms_std],
        mel_mean, mel_std
    ])
    
    return features.astype('float32')

def extract_cnn_features(audio, sr=SR, max_len=100):
    """CNN için MFCC sequence - 2D array olarak döner (timesteps, features)"""
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_fft=2048, hop_length=512)
    
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0,0), (0, max_len - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :max_len]
    
    mfcc = (mfcc - mfcc.mean()) / (mfcc.std() + 1e-8)
    # Shape: (N_MFCC, max_len) -> (max_len, N_MFCC)
    return mfcc.T.astype('float32')

# Augmentation functions
def augment_with_noise(audio, noise_factor=0.005):
    return audio + noise_factor * np.random.randn(len(audio))

def augment_with_pitch(audio, sr, n_steps=2):
    try:
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=random.uniform(-n_steps, n_steps))
    except:
        return audio

def augment_with_speed(audio, rate_range=(0.9, 1.1)):
    return librosa.effects.time_stretch(audio, rate=random.uniform(*rate_range))

augmentation_funcs = [
    lambda x: augment_with_noise(x),
    lambda x: augment_with_pitch(x, SR),
    lambda x: augment_with_speed(x),
]

# ============================================================
# 3. DATASET LOADING (MEMORY OPTIMIZED)
# ============================================================
print("\n📊 Adım 2: Dataset Yükle (Memory-Optimized)")
print("-"*120)

X_flat, X_cnn, y = [], [], []
sample_count = {label: 0 for label in class_folders.keys()}

for label, folder in class_folders.items():
    if not os.path.exists(folder):
        continue
    
    files = [f for f in os.listdir(folder) if f.endswith('.wav')]
    print(f"\n  {label}:")
    
    for file_idx, file in enumerate(files):
        try:
            audio, sr = librosa.load(os.path.join(folder, file), sr=SR)
            
            # Advanced preprocessing
            audio = advanced_noise_reduction(audio, sr)
            audio = advanced_normalization(audio)
            
            segments = segment_audio(audio, sr, overlap_ratio=0.3)
            
            for seg in segments:
                if len(seg) < SR * SEGMENT_SEC * 0.5:
                    continue
                
                # Check max samples limit
                if sample_count[label] >= MAX_SAMPLES_PER_CLASS:
                    break
                
                # Original
                feat_flat = extract_advanced_features(seg)
                feat_cnn = extract_cnn_features(seg)
                
                X_flat.append(feat_flat)
                X_cnn.append(feat_cnn)
                y.append(label)
                sample_count[label] += 1
                
                # Augmentations (only 1x, not 2x to save memory)
                aug_func = random.choice(augmentation_funcs)
                seg_aug = aug_func(seg)
                feat_flat_aug = extract_advanced_features(seg_aug)
                feat_cnn_aug = extract_cnn_features(seg_aug)
                
                X_flat.append(feat_flat_aug)
                X_cnn.append(feat_cnn_aug)
                y.append(label)
                sample_count[label] += 1
                
                if sample_count[label] >= MAX_SAMPLES_PER_CLASS:
                    break
            
            if (file_idx + 1) % 5 == 0 or file_idx == len(files) - 1:
                print(f"    ✅ {file_idx + 1}/{len(files)} dosya işlendi")
                
        except Exception as e:
            print(f"    ⚠️  {file}: {str(e)[:50]}")

print("\n  Converting to numpy arrays...")
X_flat = np.array(X_flat, dtype="float32")
X_cnn = np.array(X_cnn, dtype="float32")

print(f"  Memory usage: X_flat: {X_flat.nbytes / 1e6:.1f}MB, X_cnn: {X_cnn.nbytes / 1e6:.1f}MB")
print(f"  X_flat shape: {X_flat.shape}")
print(f"  X_cnn shape: {X_cnn.shape}")

scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat).astype('float32')

le = LabelEncoder()
y = le.fit_transform(y)

print(f"\n✅ Dataset hazır:")
print(f"   Flat Shape: {X_flat.shape}")
print(f"   CNN Shape: {X_cnn.shape}")
print(f"   Total Samples: {len(y)}")
print(f"   Classes: {le.classes_}")
print(f"   Distribution: {np.bincount(y)}")

# Train-test split
X_flat_train, X_flat_test, X_cnn_train, X_cnn_test, y_train, y_test = train_test_split(
    X_flat, X_cnn, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTrain: {len(y_train)} | Test: {len(y_test)}")
print(f"X_cnn_train shape: {X_cnn_train.shape}")
print(f"X_cnn_test shape: {X_cnn_test.shape}")

# Memory cleanup
del X_flat, X_cnn
gc.collect()

# ============================================================
# 4. RESHAPE CNN DATA (FIXED!)
# ============================================================
print("\n📊 CNN Data Reshape:")
print("-"*120)

# ✅ FIX: X_cnn_train shape: (num_samples, 100, 40) -> reshape to (num_samples, 100, 1)
# The extract_cnn_features returns (max_len, N_MFCC) = (100, 40)
# But we need to collapse the N_MFCC dimension into the channel dimension
# So reshape to (num_samples, 100, 1) and embed the 40 MFCC features as channels

print(f"  Before reshape - X_cnn_train: {X_cnn_train.shape}")

# OPTION 1: Keep features as they are (num_samples, 100, 40) - each MFCC coefficient is a channel
X_cnn_train_reshaped = X_cnn_train  # Already (num_samples, 100, 40) - perfect for Conv1D!
X_cnn_test_reshaped = X_cnn_test

print(f"  After reshape - X_cnn_train: {X_cnn_train_reshaped.shape}")
print(f"  After reshape - X_cnn_test: {X_cnn_test_reshaped.shape}")
print(f"  ✅ Shape is correct for Conv1D: (num_samples, timesteps=100, channels=40)")

# ============================================================
# 5. MODEL BUILDERS
# ============================================================

def build_mlp(input_dim, units, dropout_rate=0.3, lr=5e-4):
    model = Sequential()
    model.add(Dense(units[0], activation="relu", input_dim=input_dim))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    
    for u in units[1:]:
        model.add(Dense(u, activation="relu"))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    model.compile(optimizer=Adam(learning_rate=lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_cnn(input_shape, filters):
    """
    input_shape should be (timesteps, channels) = (100, 40)
    """
    model = Sequential()
    
    for i, f in enumerate(filters):
        kernel = 7 if i == 0 else 5 if i == 1 else 3
        model.add(Conv1D(f, kernel, padding="same", activation="relu",
                        input_shape=input_shape if i == 0 else None))
        model.add(BatchNormalization())
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.2))
    
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    
    model.compile(optimizer=Adam(learning_rate=3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, activation='relu', return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        BatchNormalization(),
        LSTM(32, activation='relu', return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    
    model.compile(optimizer=Adam(learning_rate=3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# ============================================================
# 6. TRAINING FUNCTION
# ============================================================

def train_model(X_train, X_test, y_train, y_test, model, epochs=EPOCHS, verbose=0):
    """Train model with callbacks"""
    
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weight_dict = {i: w for i, w in enumerate(class_weights)}
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    ]
    
    model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=verbose
    )
    
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    
    return f1, acc, y_pred, y_test

# ============================================================
# 7. GRID SEARCH - MLP
# ============================================================

print("\n" + "="*120)
print("🔍 Adım 3: Grid Search - Optimal Model Configurasyonları Bul")
print("="*120)

print("\n🧠 MLP - GRID SEARCH")
print("-"*120)

mlp_configs = [
    [256, 512],
    [512, 256],
    [256, 256],
    [128, 256, 512],
]

mlp_results = {}
best_mlp_f1 = 0
best_mlp_config = None

for idx, config in enumerate(mlp_configs, 1):
    print(f"  [{idx}/{len(mlp_configs)}] Testing {config}...", end=" ", flush=True)
    
    model = build_mlp(X_flat_train.shape[1], config)
    f1, acc, _, _ = train_model(X_flat_train, X_flat_test, y_train, y_test, model, verbose=0)
    
    mlp_results[str(config)] = {'f1': f1, 'acc': acc}
    print(f"F1: {f1:.4f} | Acc: {acc:.4f}")
    
    if f1 > best_mlp_f1:
        best_mlp_f1 = f1
        best_mlp_config = config
    
    # Memory cleanup
    del model
    gc.collect()

# ============================================================
# 8. GRID SEARCH - CNN (FIXED!)
# ============================================================

print("\n🧠 CNN - GRID SEARCH")
print("-"*120)

cnn_configs = [
    [64, 128, 256],
    [128, 256, 256],
    [128, 256, 128],
]

cnn_results = {}
best_cnn_f1 = 0
best_cnn_config = None

for idx, config in enumerate(cnn_configs, 1):
    print(f"  [{idx}/{len(cnn_configs)}] Testing {config}...", end=" ", flush=True)
    
    # ✅ FIX: Pass (100, 40) as input_shape
    input_shape = X_cnn_train_reshaped.shape[1:]  # This is (100, 40)
    model = build_cnn(input_shape, config)
    f1, acc, _, _ = train_model(X_cnn_train_reshaped, X_cnn_test_reshaped, y_train, y_test, model, verbose=0)
    
    cnn_results[str(config)] = {'f1': f1, 'acc': acc}
    print(f"F1: {f1:.4f} | Acc: {acc:.4f}")
    
    if f1 > best_cnn_f1:
        best_cnn_f1 = f1
        best_cnn_config = config
    
    # Memory cleanup
    del model
    gc.collect()

# ============================================================
# 9. LSTM MODEL
# ============================================================

print("\n🧠 LSTM - TRAINING")
print("-"*120)

model_lstm = build_lstm(X_cnn_train_reshaped.shape[1:])
f1_lstm, acc_lstm, _, _ = train_model(X_cnn_train_reshaped, X_cnn_test_reshaped, y_train, y_test, model_lstm, verbose=0)
print(f"  LSTM F1: {f1_lstm:.4f} | Acc: {acc_lstm:.4f}")

# ============================================================
# 10. GRID SEARCH - SVM
# ============================================================

print("\n🧠 SVM - GRID SEARCH")
print("-"*120)

svm_results = {}
best_svm_f1 = 0
best_svm_params = None

svm_configs = [
    {'C': 1.0, 'kernel': 'rbf'},
    {'C': 5.0, 'kernel': 'rbf'},
    {'C': 10.0, 'kernel': 'rbf'},
    {'C': 1.0, 'kernel': 'poly'},
]

for idx, cfg in enumerate(svm_configs, 1):
    print(f"  [{idx}/{len(svm_configs)}] Testing {cfg}...", end=" ", flush=True)
    
    svm = SVC(C=cfg['C'], kernel=cfg['kernel'], random_state=RANDOM_STATE, probability=True)
    svm.fit(X_flat_train, y_train)
    
    y_pred = svm.predict(X_flat_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    acc = accuracy_score(y_test, y_pred)
    
    svm_results[str(cfg)] = {'f1': f1, 'acc': acc}
    print(f"F1: {f1:.4f} | Acc: {acc:.4f}")
    
    if f1 > best_svm_f1:
        best_svm_f1 = f1
        best_svm_params = cfg

# ============================================================
# 11. KNN OPTIMIZATION
# ============================================================

print("\n🧠 KNN - OPTIMIZATION")
print("-"*120)

knn_results = {}
best_knn_f1 = 0
best_knn_k = 5

for k in range(3, 16):  # REDUCED from 20
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_flat_train, y_train)
    y_pred = knn.predict(X_flat_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    acc = accuracy_score(y_test, y_pred)
    
    knn_results[f'k={k}'] = {'f1': f1, 'acc': acc}
    
    if f1 > best_knn_f1:
        best_knn_f1 = f1
        best_knn_k = k
    
    if k in [3, 5, 10, 15]:
        print(f"  k={k:2d} → F1: {f1:.4f} | Acc: {acc:.4f}")

print(f"  Best KNN: k={best_knn_k} (F1: {best_knn_f1:.4f})")

# ============================================================
# 12. BUILD BEST INDIVIDUAL MODELS
# ============================================================

print("\n" + "="*120)
print("🎯 Adım 4: En İyi Modelleri Oluştur")
print("="*120)

# Best MLP
model_mlp_best = build_mlp(X_flat_train.shape[1], best_mlp_config)
f1_mlp, acc_mlp, _, _ = train_model(X_flat_train, X_flat_test, y_train, y_test, model_mlp_best, verbose=0)
print(f"\n✅ Best MLP: {best_mlp_config} → F1: {f1_mlp:.4f}")

# Best CNN
model_cnn_best = build_cnn(X_cnn_train_reshaped.shape[1:], best_cnn_config)
f1_cnn, acc_cnn, _, _ = train_model(X_cnn_train_reshaped, X_cnn_test_reshaped, y_train, y_test, model_cnn_best, verbose=0)
print(f"✅ Best CNN: {best_cnn_config} → F1: {f1_cnn:.4f}")

# Best SVM
svm_best = SVC(C=best_svm_params['C'], kernel=best_svm_params['kernel'], random_state=RANDOM_STATE, probability=True)
svm_best.fit(X_flat_train, y_train)
y_pred_svm = svm_best.predict(X_flat_test)
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"✅ Best SVM: {best_svm_params} → F1: {f1_svm:.4f}")

# Best KNN
knn_best = KNeighborsClassifier(n_neighbors=best_knn_k)
knn_best.fit(X_flat_train, y_train)
y_pred_knn = knn_best.predict(X_flat_test)
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"✅ Best KNN: k={best_knn_k} → F1: {f1_knn:.4f}")

# ============================================================
# 13. ENSEMBLE METHODS
# ============================================================

print("\n" + "="*120)
print("🎯 Adım 5: Ensemble Methods")
print("="*120)

# Voting Classifier
print("\n📊 VOTING CLASSIFIER (Soft Voting)")
voting_clf = VotingClassifier(
    estimators=[
        ('svm', svm_best),
        ('knn', knn_best),
    ],
    voting='soft'
)
voting_clf.fit(X_flat_train, y_train)
y_pred_voting = voting_clf.predict(X_flat_test)
f1_voting = f1_score(y_test, y_pred_voting, average='weighted')
acc_voting = accuracy_score(y_test, y_pred_voting)
print(f"  → F1: {f1_voting:.4f} | Acc: {acc_voting:.4f}")

# Stacking Classifier
print("\n📊 STACKING CLASSIFIER")
stacking_clf = StackingClassifier(
    estimators=[
        ('svm', svm_best),
        ('knn', knn_best),
    ],
    final_estimator=SVC(kernel='rbf', C=1, probability=True, random_state=RANDOM_STATE),
    cv=3  # REDUCED from 5
)
stacking_clf.fit(X_flat_train, y_train)
y_pred_stacking = stacking_clf.predict(X_flat_test)
f1_stacking = f1_score(y_test, y_pred_stacking, average='weighted')
acc_stacking = accuracy_score(y_test, y_pred_stacking)
print(f"  → F1: {f1_stacking:.4f} | Acc: {acc_stacking:.4f}")

# Neural Network Ensemble
print("\n📊 NEURAL NETWORK ENSEMBLE (MLP + CNN + LSTM)")

def predict_nn_ensemble(X_flat_test_input, X_cnn_test_input):
    mlp_pred = model_mlp_best.predict(X_flat_test_input, verbose=0)
    cnn_pred = model_cnn_best.predict(X_cnn_test_input, verbose=0)
    lstm_pred = model_lstm.predict(X_cnn_test_input, verbose=0)
    
    ensemble_pred = (mlp_pred + cnn_pred + lstm_pred) / 3
    return np.argmax(ensemble_pred, axis=1)

y_pred_nn_ensemble = predict_nn_ensemble(X_flat_test, X_cnn_test_reshaped)
f1_nn_ensemble = f1_score(y_test, y_pred_nn_ensemble, average='weighted')
acc_nn_ensemble = accuracy_score(y_test, y_pred_nn_ensemble)
print(f"  → F1: {f1_nn_ensemble:.4f} | Acc: {acc_nn_ensemble:.4f}")

# ============================================================
# 14. FINAL RESULTS
# ============================================================

print("\n" + "="*120)
print("🏆 FINAL RANKING - HER MODELIN BEST PERFORMANSI")
print("="*120)

final_results = {
    'MLP': {'f1': f1_mlp, 'acc': acc_mlp, 'config': str(best_mlp_config)},
    'CNN': {'f1': f1_cnn, 'acc': acc_cnn, 'config': str(best_cnn_config)},
    'LSTM': {'f1': f1_lstm, 'acc': acc_lstm, 'config': 'Default LSTM'},
    'SVM': {'f1': f1_svm, 'acc': acc_svm, 'config': str(best_svm_params)},
    'KNN': {'f1': f1_knn, 'acc': acc_knn, 'config': f'k={best_knn_k}'},
    'Voting': {'f1': f1_voting, 'acc': acc_voting, 'config': 'SVM + KNN (soft)'},
    'Stacking': {'f1': f1_stacking, 'acc': acc_stacking, 'config': 'SVM + KNN + SVM meta'},
    'NN_Ensemble': {'f1': f1_nn_ensemble, 'acc': acc_nn_ensemble, 'config': 'MLP + CNN + LSTM'},
}

ranking = sorted(final_results.items(), key=lambda x: x[1]['f1'], reverse=True)

print("\n{:<3} {:<20} {:<12} {:<12} {:<50}".format("Rank", "Model", "F1 Score", "Accuracy", "Config"))
print("-"*120)

for rank, (model_name, info) in enumerate(ranking, 1):
    config_str = info['config'][:45] + "..." if len(info['config']) > 45 else info['config']
    print("{:<3} {:<20} {:<12.4f} {:<12.4f} {:<50}".format(
        rank, model_name, info['f1'], info['acc'], config_str
    ))

best_model = ranking[0]
print("\n" + "="*120)
print(f"🎯 BEST MODEL: {best_model[0]}")
print(f"   F1 Score: {best_model[1]['f1']:.4f}")
print(f"   Accuracy: {best_model[1]['acc']:.4f}")
print(f"   Config: {best_model[1]['config']}")
print("="*120)

# ============================================================
# 15. DETAILED ANALYSIS
# ============================================================

print("\n" + "="*120)
print(f"📊 DETAILED ANALYSIS: {best_model[0]}")
print("="*120)

# Get best model predictions
if best_model[0] == 'MLP':
    y_pred_final = np.argmax(model_mlp_best.predict(X_flat_test, verbose=0), axis=1)
elif best_model[0] == 'CNN':
    y_pred_final = np.argmax(model_cnn_best.predict(X_cnn_test_reshaped, verbose=0), axis=1)
elif best_model[0] == 'LSTM':
    y_pred_final = np.argmax(model_lstm.predict(X_cnn_test_reshaped, verbose=0), axis=1)
elif best_model[0] == 'NN_Ensemble':
    y_pred_final = predict_nn_ensemble(X_flat_test, X_cnn_test_reshaped)
elif best_model[0] == 'SVM':
    y_pred_final = svm_best.predict(X_flat_test)
elif best_model[0] == 'KNN':
    y_pred_final = knn_best.predict(X_flat_test)
elif best_model[0] == 'Voting':
    y_pred_final = voting_clf.predict(X_flat_test)
elif best_model[0] == 'Stacking':
    y_pred_final = stacking_clf.predict(X_flat_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=le.classes_))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_final)
print(cm)

# ============================================================
# 16. SAVE RESULTS
# ============================================================

print("\n" + "="*120)
print("💾 SONUÇLARI KAYDET")
print("="*120)

results_list = []
for model_name, info in final_results.items():
    results_list.append({
        'Model': model_name,
        'F1_Score': info['f1'],
        'Accuracy': info['acc'],
        'Config': info['config']
    })

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('F1_Score', ascending=False).reset_index(drop=True)

results_df.to_csv("lung_sound_advanced_results.csv", index=False)
print("\n✅ lung_sound_advanced_results.csv kaydedildi")

with open("best_model_config.txt", "w") as f:
    f.write("="*80 + "\n")
    f.write(f"BEST MODEL: {best_model[0]}\n")
    f.write(f"F1 Score: {best_model[1]['f1']:.4f}\n")
    f.write(f"Accuracy: {best_model[1]['acc']:.4f}\n")
    f.write("="*80 + "\n\n")
    
    f.write("ALL RESULTS:\n")
    f.write(results_df.to_string())
    f.write("\n\n" + "="*80 + "\n")

print("✅ best_model_config.txt kaydedildi")

# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "="*120)
print("✅ ADVANCED OPTIMIZATION TAMAMLANDI!")
print("="*120)

print("\n📊 ÖZET:")
print(f"  • Toplam ses dosyası: {total_files}")
print(f"  • Toplam samples (limited to {MAX_SAMPLES_PER_CLASS}/class): {len(y_train) + len(y_test)}")
print(f"  • Train/Test Split: {len(y_train)}/{len(y_test)}")
print(f"  • Test edilen model sayısı: {len(results_df)}")
print(f"  • En iyi model: {best_model[0]} (F1: {best_model[1]['f1']:.4f})")

print("\n🔧 YAPILAN OPTİMİZASYONLAR:")
print("  ✓ Advanced noise reduction (spectral gating)")
print("  ✓ Peak & RMS normalization")
print("  ✓ Rich feature extraction (280+ features)")
print("  ✓ Memory-optimized data handling")
print("  ✓ High-overlap segmentation")
print("  ✓ Grid Search hyperparameter tuning")
print("  ✓ Multiple ensemble methods")
print("  ✓ Neural network ensemble")
print("  ✓ Balanced class weights")
print("  ✓ FIXED: Correct CNN data shape (num_samples, timesteps=100, channels=40)")

print("\n📁 ÇIKTILARI:")
print("  ✓ lung_sound_advanced_results.csv")
print("  ✓ best_model_config.txt")

print("\n" + "="*120 + "\n")


🫁 LUNG SOUND CLASSIFICATION - CNN FIX + KAN + MODEL PERSISTENCE

📂 Adım 1: Ses Dosyalarını Yükle
------------------------------------------------------------------------------------------------------------------------
  📁 Asthma: 96 dosya
  📁 COPD: 112 dosya
  📁 Healthy: 112 dosya

✅ Toplam 320 ses dosyası bulundu

📊 Adım 2: Dataset Yükle (Memory-Optimized)
------------------------------------------------------------------------------------------------------------------------

  Asthma:
    ✅ 5/96 dosya işlendi
    ✅ 10/96 dosya işlendi
    ✅ 15/96 dosya işlendi
    ✅ 20/96 dosya işlendi
    ✅ 25/96 dosya işlendi
    ✅ 30/96 dosya işlendi
    ✅ 35/96 dosya işlendi
    ✅ 40/96 dosya işlendi
    ✅ 45/96 dosya işlendi
    ✅ 50/96 dosya işlendi
    ✅ 55/96 dosya işlendi
    ✅ 60/96 dosya işlendi
    ✅ 65/96 dosya işlendi
    ✅ 70/96 dosya işlendi
    ✅ 75/96 dosya işlendi
    ✅ 80/96 dosya işlendi
    ✅ 85/96 dosya işlendi
    ✅ 90/96 dosya işlendi
    ✅ 95/96 dosya işlendi
    ✅ 96/96 do

OSError: Cannot save file into a non-existent directory: '\mnt\user-data\outputs'